# Week 2 · Linear algebra revision

You logged that you'd forgotten a lot of linear algebra. This notebook rebuilds exactly the parts that ML uses, in the order regression needs them: vectors and the dot product → matrix multiplication → inverses and solving $Ax = b$ → projections. The last one leads straight into least squares (notebook `04`). Every section follows the same pattern: what it is, the formula, where it comes from, one worked example with every number computed, its properties, where it's used, and how it connects to the next section. **Supplementary reading** if a section still feels thin: 3Blue1Brown, *Essence of Linear Algebra*, videos 1–4 and 7–9 (about 10 minutes each, very visual), and *Mathematics for Machine Learning* Ch. 2–3.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **vector $x \in \mathbb{R}^n$**: An ordered list of $n$ numbers $x = (x_1, \dots, x_n)$. In ML, one sample's features.
- **matrix $A \in \mathbb{R}^{m \times n}$**: A grid of numbers with $m$ rows and $n$ columns. $A_{ij}$ is the entry in row $i$, column $j$.
- **transpose $A^\top$**: Rows become columns: $(A^\top)_{ij} = A_{ji}$. An $m \times n$ matrix becomes $n \times m$.
- **norm $\lVert x \rVert$**: The length of a vector: $\lVert x \rVert = \sqrt{x_1^2 + \dots + x_n^2}$.
- **identity $I$**: The square matrix with 1s on the diagonal and 0s elsewhere. $IA = AI = A$, the matrix version of the number 1.
- **linearly independent**: No column can be written as a combination of the others. For a square matrix: the matrix has an inverse.

## 1. Vectors and the dot product

**The dot product measures how much two vectors point in the same direction, scaled by their lengths.**

**The setting.** Two vectors with the same number of entries, $u, v \in \mathbb{R}^n$. In ML, $u$ is often a weight vector $w$ and $v$ is one sample's features $x$.

$$
u \cdot v = u^\top v = \sum_{i=1}^{n} u_i v_i = u_1 v_1 + u_2 v_2 + \dots + u_n v_n
$$

Read it as: multiply matching entries, then add everything up. The result is **one number**, not a vector.

**Where the geometry comes from.** The same number can be written with lengths and the angle $\theta$ between the vectors:

$$
u \cdot v = \lVert u \rVert \, \lVert v \rVert \cos\theta \qquad\Longrightarrow\qquad \cos\theta = \frac{u \cdot v}{\lVert u \rVert\, \lVert v \rVert}
$$

So the sign tells you the direction: $\cos\theta > 0$ means the angle is under $90°$, $\cos\theta = 0$ means they're perpendicular (**orthogonal**), and $\cos\theta < 0$ means the angle is over $90°$.

**Worked example.** $u = (1, 2, 2)$ and $v = (2, 0, 1)$:

| $i$ | $u_i$ | $v_i$ | $u_i v_i$ |
|---|---|---|---|
| 1 | 1 | 2 | 2 |
| 2 | 2 | 0 | 0 |
| 3 | 2 | 1 | 2 |
| **sum** |  |  | **4** |

Lengths: $\lVert u \rVert = \sqrt{1 + 4 + 4} = 3$ and $\lVert v \rVert = \sqrt{4 + 0 + 1} = \sqrt{5} \approx 2.236$. So $\cos\theta = \frac{4}{3 \times 2.236} \approx 0.596$ and $\theta \approx 53.4°$. Notice the middle term is 0 because $v_2 = 0$: that coordinate contributes nothing to the alignment.

In [ ]:
import numpy as np
u = np.array([1, 2, 2]); v = np.array([2, 0, 1])
cos = (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))
print(u @ v, round(cos, 3), round(np.degrees(np.arccos(cos)), 1))   # → 4 0.596 53.4

**Key properties.**

| Property | Formula |
|---|---|
| symmetric | $u \cdot v = v \cdot u$ |
| length from itself | $u \cdot u = \lVert u \rVert^2$ |
| orthogonal | $u \cdot v = 0 \iff$ perpendicular |
| linear | $u \cdot (a v + b w) = a\,(u \cdot v) + b\,(u \cdot w)$ |

**Where it's used.**

| Use | How |
|---|---|
| linear and logistic regression | the prediction is $w \cdot x + b$ |
| neural networks | every neuron computes a dot product, then a nonlinearity |
| attention (Week 8) | query $\cdot$ key measures how related two tokens are |
| similarity search, RAG | cosine similarity $\cos\theta$ between embedding vectors |

**Connection to the next section:** a matrix times a vector is just **many dot products at once**, one per row.

**Quick check:** $u = (3, 4)$ and $v = (4, -3)$. What is $u \cdot v$, and what does it tell you?

<details><summary>Answer</summary>

$12 - 12 = 0$, so they're orthogonal (perpendicular).

</details>

## 2. Matrix multiplication

**Matrix multiplication combines two linear maps into one: each entry of $AB$ is a row of $A$ dotted with a column of $B$.**

**The setting.** $A \in \mathbb{R}^{m \times n}$ and $B \in \mathbb{R}^{n \times p}$. The inner sizes must match (both $n$), and the result is $m \times p$.

$$
(AB)_{ij} = \sum_{k=1}^{n} A_{ik} B_{kj} = (\text{row } i \text{ of } A) \cdot (\text{column } j \text{ of } B)
$$

**The same product seen two ways.** (1) Entry by entry, as above. (2) Column by column: $Ax$ is a **weighted sum of the columns of $A$**, with weights $x_1, \dots, x_n$. View (2) is the one that explains regression and projections:

$$
A x = x_1 \begin{pmatrix} A_{11} \\ \vdots \\ A_{m1} \end{pmatrix} + x_2 \begin{pmatrix} A_{12} \\ \vdots \\ A_{m2} \end{pmatrix} + \dots + x_n \begin{pmatrix} A_{1n} \\ \vdots \\ A_{mn} \end{pmatrix}
$$

**Worked example.** $A = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}$ and $B = \begin{pmatrix} 2 & 0 \\ 1 & 3 \end{pmatrix}$:

| Entry | Row of $A$ · column of $B$ | Value |
|---|---|---|
| $(AB)_{11}$ | $(1, 2) \cdot (2, 1) = 1 \cdot 2 + 2 \cdot 1$ | 4 |
| $(AB)_{12}$ | $(1, 2) \cdot (0, 3) = 1 \cdot 0 + 2 \cdot 3$ | 6 |
| $(AB)_{21}$ | $(3, 4) \cdot (2, 1) = 3 \cdot 2 + 4 \cdot 1$ | 10 |
| $(AB)_{22}$ | $(3, 4) \cdot (0, 3) = 3 \cdot 0 + 4 \cdot 3$ | 12 |

So $AB = \begin{pmatrix} 4 & 6 \\ 10 & 12 \end{pmatrix}$. In the other order, $BA = \begin{pmatrix} 2 & 4 \\ 10 & 14 \end{pmatrix}$, which is different. **The order matters.**

In [ ]:
A = np.array([[1, 2], [3, 4]]); B = np.array([[2, 0], [1, 3]])
print(A @ B)
# → [[ 4  6]
#    [10 12]]
print(B @ A)
# → [[ 2  4]
#    [10 14]]
x = np.array([1, 10])
print(A @ x, 1 * A[:, 0] + 10 * A[:, 1])    # → [21 43] [21 43]   Ax = weighted sum of columns

**Key properties.**

| Property | Formula | Note |
|---|---|---|
| not commutative | $AB \ne BA$ in general | even when both are defined |
| associative | $(AB)C = A(BC)$ | you can choose which product to do first |
| transpose of a product | $(AB)^\top = B^\top A^\top$ | the order flips |
| identity | $IA = AI = A$ |  |

**Where the transpose rule comes from.** Entry $(i, j)$ of $(AB)^\top$ is entry $(j, i)$ of $AB$, which is $\sum_k A_{jk} B_{ki} = \sum_k (B^\top)_{ik} (A^\top)_{kj}$. That last sum is entry $(i, j)$ of $B^\top A^\top$.

In [ ]:
print(np.array_equal((A @ B).T, B.T @ A.T))    # → True

**Where it's used:** a whole dataset's predictions at once, $\hat y = Xw$ with $X \in \mathbb{R}^{n \times d}$; every layer of a neural network, $h = Wx + b$; and chaining transformations.

**Connection:** if $A x = b$ maps $x$ to $b$, can we go back from $b$ to $x$? That's the inverse.

**Quick check:** What is the shape of $X^\top X$ when $X$ is $100 \times 3$?

<details><summary>Answer</summary>

$(3 \times 100)(100 \times 3) = 3 \times 3$. It's small and square, which is why the normal equation is cheap to solve.

</details>

## 3. Inverses and solving $Ax = b$

**The inverse $A^{-1}$ undoes $A$: if $Ax = b$, then $x = A^{-1} b$.**

**The setting.** $A$ must be square, $n \times n$, and its columns must be linearly independent. Then $A^{-1}$ exists and $A^{-1} A = A A^{-1} = I$.

**The $2 \times 2$ formula** (worth memorizing for the exam). For $A = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$:

$$
A^{-1} = \frac{1}{ad - bc} \begin{pmatrix} d & -b \\ -c & a \end{pmatrix} \qquad \det A = ad - bc
$$

Read it as: swap the diagonal, negate the off-diagonal, divide by the determinant. If $\det A = 0$ there is no inverse.

**Worked example.** Solve $\begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix} x = \begin{pmatrix} 5 \\ 11 \end{pmatrix}$. First $\det A = 1 \cdot 4 - 2 \cdot 3 = -2$, so:

$$
A^{-1} = \frac{1}{-2} \begin{pmatrix} 4 & -2 \\ -3 & 1 \end{pmatrix} = \begin{pmatrix} -2 & 1 \\ 1.5 & -0.5 \end{pmatrix}
$$

$$
x = A^{-1} b = \begin{pmatrix} -2 \cdot 5 + 1 \cdot 11 \\ 1.5 \cdot 5 - 0.5 \cdot 11 \end{pmatrix} = \begin{pmatrix} -10 + 11 \\ 7.5 - 5.5 \end{pmatrix} = \begin{pmatrix} 1 \\ 2 \end{pmatrix}
$$

Check: $1 \cdot 1 + 2 \cdot 2 = 5$ and $3 \cdot 1 + 4 \cdot 2 = 11$. Both match.

In [ ]:
A = np.array([[1.0, 2.0], [3.0, 4.0]]); b = np.array([5.0, 11.0])
print(np.linalg.inv(A))
# → [[-2.   1. ]
#    [ 1.5 -0.5]]
print(np.linalg.solve(A, b))    # → [1. 2.]   solve() is faster and more accurate than inv(A) @ b

**When there's no inverse.** $\begin{pmatrix} 1 & 2 \\ 2 & 4 \end{pmatrix}$ has $\det = 4 - 4 = 0$. Its second column is exactly 2 × the first, so the columns are **linearly dependent**: the matrix squashes the whole plane onto a line, and you can't undo that. The **rank** (the number of independent columns) is 1, not 2.

In [ ]:
S = np.array([[1.0, 2.0], [2.0, 4.0]])
print(np.linalg.det(S), np.linalg.matrix_rank(S))    # → 0.0 1

**Key properties.**

| Property | Formula |
|---|---|
| inverse of a product | $(AB)^{-1} = B^{-1} A^{-1}$ (order flips, like the transpose) |
| inverse of a transpose | $(A^\top)^{-1} = (A^{-1})^\top$ |
| exists exactly when | $\det A \ne 0$, i.e. the columns are independent (full rank) |

**Where it's used:** the normal equation $\hat w = (X^\top X)^{-1} X^\top y$ needs $X^\top X$ to be invertible. It fails when two features are copies of each other (perfect **multicollinearity**). Ridge regression fixes this by adding $\lambda I$, which makes the matrix invertible again (notebook `04`).

**Connection:** most real systems have **more equations than unknowns** (100 samples, 3 weights), so there's no exact solution. The best we can do is the closest point, and that's a projection.

**Drill (on paper).** Find the inverse of $\begin{pmatrix} 2 & 1 \\ 5 & 3 \end{pmatrix}$ and use it to solve $Ax = (3, 8)$.

<details><summary>Worked answer</summary>

$\det = 2 \cdot 3 - 1 \cdot 5 = 1$, so $A^{-1} = \begin{pmatrix} 3 & -1 \\ -5 & 2 \end{pmatrix}$. Then $x = (3 \cdot 3 - 1 \cdot 8,\ -5 \cdot 3 + 2 \cdot 8) = (1, 1)$. Check: $2 + 1 = 3$ and $5 + 3 = 8$.

</details>

## 4. Projections: the closest point

**Projecting $b$ onto a direction $a$ gives the point on that line closest to $b$; the leftover piece is perpendicular to the line.**

**The setting.** A vector $b$ and a direction $a$, both in $\mathbb{R}^n$. We want the multiple $\hat c\, a$ that is closest to $b$.

$$
\operatorname{proj}_a(b) = \hat c\, a \quad\text{with}\quad \hat c = \frac{a \cdot b}{a \cdot a}
$$

**Where it comes from.** The closest point is the one where the error $e = b - \hat c a$ is **perpendicular** to $a$. Write that condition and solve it one step at a time:

$$
a \cdot (b - \hat c\, a) = 0 \;\Longrightarrow\; a \cdot b - \hat c\,(a \cdot a) = 0 \;\Longrightarrow\; \hat c = \frac{a \cdot b}{a \cdot a}
$$

**Worked example.** $b = (2, 0)$ and $a = (1, 1)$. Then $a \cdot b = 2$ and $a \cdot a = 2$, so $\hat c = 1$ and the projection is $(1, 1)$. The error is $e = (2, 0) - (1, 1) = (1, -1)$, and indeed $a \cdot e = 1 - 1 = 0$: it's perpendicular.

In [ ]:
a = np.array([1.0, 1.0]); b = np.array([2.0, 0.0])
c = (a @ b) / (a @ a)
proj = c * a; e = b - proj
print(proj, e, a @ e)      # → [1. 1.] [ 1. -1.] 0.0

**From one direction to many: this is least squares.** Now replace the single direction $a$ with the **columns of a data matrix** $X$. The predictions $Xw$ are combinations of those columns (view 2 of matrix multiplication). The best $w$ makes the error $y - Xw$ perpendicular to **every** column of $X$, which says $X^\top (y - Xw) = 0$. Expand one step:

$$
X^\top (y - X\hat w) = 0 \;\Longrightarrow\; X^\top y - X^\top X \hat w = 0 \;\Longrightarrow\; \boxed{X^\top X\, \hat w = X^\top y}
$$

That's the **normal equation** from Week 1, now with a picture: least squares **projects $y$ onto the space spanned by the feature columns**. "Normal" is an old word for perpendicular, and that's where the name comes from. With one column $a$, it reduces exactly to $\hat c = \frac{a \cdot b}{a \cdot a}$.

In [ ]:
X = np.array([[1.0, 1.0], [1.0, 2.0], [1.0, 3.0], [1.0, 4.0]])   # a column of 1s (intercept) + one feature
y = np.array([2.0, 3.0, 5.0, 4.0])
w = np.linalg.solve(X.T @ X, X.T @ y)
print(w)                          # → [1.5 0.8]   intercept 1.5, slope 0.8
print(np.allclose(X.T @ (y - X @ w), 0))   # → True   the error is perpendicular to both columns

**Key properties.**

| Property | Meaning |
|---|---|
| error ⟂ subspace | $X^\top e = 0$; in particular the residuals sum to 0 when $X$ has a column of 1s |
| projecting twice changes nothing | a point already on the line stays where it is |
| closest point | no other $w$ gives a smaller $\lVert y - Xw \rVert$ |

**Where it's used:** linear regression (notebook `04`), PCA (Week 4 projects data onto the top eigenvectors), and Gram–Schmidt / QR decomposition (how libraries solve least squares stably).

**Connection:** notebook `04` uses exactly this $X$ and $y$ as its worked example, and gets the same line $\hat y = 1.5 + 0.8x$ by hand.

**Drill (on paper).** Project $b = (1, 2, 3)$ onto $a = (1, 1, 1)$. What is $\hat c$, and what does it equal in words?

<details><summary>Worked answer</summary>

$a \cdot b = 6$ and $a \cdot a = 3$, so $\hat c = 2$ and the projection is $(2, 2, 2)$. $\hat c$ is the **mean** of $b$: the best constant prediction is the average. That's regression with only an intercept.

</details>

---
Next: `02_map_and_entropy_in_depth.ipynb`, which covers the two Week 1 topics your quiz log flagged (MAP and entropy / cross-entropy / KL).